# Random forest

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.multiclass import type_of_target
from sklearn.ensemble import RandomForestClassifier

In [ ]:
CSV_FILE = "data/student_pass_prediction.csv"
TARGET = "passed"
FEATURES = [
    "study_hours_per_day",
    "attendance_percent",
    "previous_exam_score",
    "assignments_completed_percent",
    "sleep_hours_per_night",
]
CATEGORICAL_COLUMNS = []
FOLDS = 10
NEW_DATA_FILE = None
CSV_OPTIONS = {"sep": ",", "encoding": "utf-8-sig"}

folder = (Path('random_forest') if Path('random_forest').is_dir() else Path(".")).resolve()
results = folder / "results"
results.mkdir(exist_ok=True)

def load_csv(filename):
    path = (folder / Path(filename).expanduser()).resolve()
    if not path.is_file():
        raise FileNotFoundError(f"CSV not found: {path}\nPaths are relative to: {folder.resolve()}")
    print("Reading:", path)
    return pd.read_csv(path, **CSV_OPTIONS)

## Data

In [ ]:
df = load_csv(CSV_FILE)
print("Shape:", df.shape)
display(df.head())
display(df.isna().sum().to_frame("Missing values"))
display(df[TARGET].value_counts(dropna=False).to_frame("Class count"))

columns = FEATURES if FEATURES is not None else [c for c in df if c != TARGET]
if not columns or TARGET in columns or len(columns) != len(set(columns)):
    raise ValueError("Choose unique input columns without the target.")
X = df[columns].replace([np.inf, -np.inf], np.nan)
y = df[TARGET]
if y.isna().any() or y.nunique() < 2:
    raise ValueError("The target needs at least two classes and no missing labels.")
if type_of_target(y) not in ("binary", "multiclass"):
    raise ValueError("The target must contain class labels, not continuous values.")
if y.value_counts().min() < FOLDS:
    raise ValueError(f"Each class needs at least {FOLDS} rows for this number of folds.")
print("Inputs:", list(X.columns))

## Preprocessing

In [ ]:
if set(CATEGORICAL_COLUMNS) - set(X.columns):
    raise ValueError("Categorical columns must be selected input columns.")
numeric = [c for c in X.select_dtypes(include="number") if c not in CATEGORICAL_COLUMNS]
categorical = [c for c in X if c not in numeric]
# Fit these steps separately in each training fold.
preprocessor = ColumnTransformer([
    ("numeric", make_pipeline(
        SimpleImputer(strategy="median", keep_empty_features=True), StandardScaler()
    ), numeric),
    ("categorical", make_pipeline(
        SimpleImputer(strategy="most_frequent", keep_empty_features=True),
        OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    ), categorical),
])
cv = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)

## Cross validation

In [ ]:
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

model = make_pipeline(preprocessor, classifier)
scores = cross_validate(model, X, y, cv=cv, scoring="accuracy", error_score="raise")["test_score"]
fold_scores = pd.DataFrame({"Fold": range(1, FOLDS + 1), "Accuracy": scores})
display(fold_scores)
fold_scores.to_csv(results / "fold_scores.csv", index=False)

## Confusion matrix

In [ ]:
predictions = cross_val_predict(model, X, y, cv=cv)
labels = np.unique(y)
matrix = pd.DataFrame(confusion_matrix(y, predictions, labels=labels),
                      index=pd.Index(labels, name="Actual"),
                      columns=pd.Index(labels, name="Predicted"))
display(matrix)
matrix.to_csv(results / "confusion_matrix.csv")
pd.DataFrame({"Actual": y, "Predicted": predictions}).to_csv(
    results / "cross_validation_predictions.csv", index=False
)

## Results

In [ ]:
summary = pd.DataFrame({"Mean accuracy": [scores.mean()], "Standard deviation": [scores.std()]})
display(summary)
summary.to_csv(results / "summary.csv", index=False)
# Refit using all labelled rows.
best_model = clone(model).fit(X, y)

## Save and load

In [ ]:
joblib.dump(best_model, results / "best_model.pkl")
loaded_model = joblib.load(results / "best_model.pkl")

## New predictions

In [ ]:
if NEW_DATA_FILE:
    new_data = load_csv(NEW_DATA_FILE)
    inputs = new_data.loc[:, loaded_model.feature_names_in_].replace([np.inf, -np.inf], np.nan)
    new_data["Prediction"] = loaded_model.predict(inputs)
    display(new_data)
    new_data.to_csv(results / "new_predictions.csv", index=False)
else:
    print("No new data supplied.")